In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
KEGG Pathway and Element Data Collection for Transport Genes
This script collects KEGG pathways, networks, and elements for a given set of human transport-related genes,
then extracts detailed element information and saves it as a JSON file.
"""

import pandas as pd
import gc
import numpy as np
import networkx as nx
import re
import json
import requests
import warnings
import time
from collections import OrderedDict
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

warnings.simplefilter(action='ignore', category=FutureWarning)
gc.collect()

# ----------------------------------------------------------------------
# 1. KEGG API helper with retry logic
# ----------------------------------------------------------------------
def get_kegg_info(kegg_ref):
    url = f"http://rest.kegg.jp/get/{kegg_ref}"
    session = requests.Session()
    retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    session.mount("http://", HTTPAdapter(max_retries=retries))
    try:
        response = session.get(url, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {kegg_ref}: {e}")
        return None

# ----------------------------------------------------------------------
# 2. Target genes (transport-related)
# ----------------------------------------------------------------------
transport_genes_dict = {
    # ABC transporters
    'ABCB1': 'hsa:5243',
    'ABCC1': 'hsa:4363',
    'ABCC2': 'hsa:1244',
    'ABCC3': 'hsa:8714',
    'ABCC4': 'hsa:10257',
    'ABCC5': 'hsa:10057',
    'ABCC6': 'hsa:368',
    'ABCG2': 'hsa:9429',
    'CFTR': 'hsa:1080',
    'ABCA1': 'hsa:19',
    'ABCG5': 'hsa:64240',
    'ABCG8': 'hsa:64241',
    # SLC family
    'SLC2A1': 'hsa:6513',
    'SLC5A1': 'hsa:6523',
    'SLC6A4': 'hsa:6532',
    'SLC12A2': 'hsa:6558',
    'SLC15A1': 'hsa:6564',
    'SLC22A4': 'hsa:6583',
    'SLC22A5': 'hsa:6584',
    'SLC22A6': 'hsa:9120',
    'SLC22A7': 'hsa:10864',
    'SLC22A8': 'hsa:9376',
    'SLC25A4': 'hsa:291',
    'SLC47A1': 'hsa:55244',
    # Ion channels
    'SCN1A': 'hsa:6323',
    'SCN5A': 'hsa:6331',
    'KCNJ11': 'hsa:3767',
    'CACNA1C': 'hsa:775',
    'CLCN1': 'hsa:1180',
    'CLCN2': 'hsa:1181',
    'TRPV1': 'hsa:7442',
    'TRPM6': 'hsa:140803',
    # Aquaporins
    'AQP1': 'hsa:358',
    'AQP2': 'hsa:359',
    'AQP3': 'hsa:360',
    'AQP4': 'hsa:361',
    # ATPases
    'ATP1A1': 'hsa:476',
    'ATP4A': 'hsa:495',
}

# Combine all genes into a single dictionary
full_dict = transport_genes_dict.copy()

# ----------------------------------------------------------------------
# 3. Extract pathways, networks, elements for each gene
# ----------------------------------------------------------------------
def get_pathways_networks_elements(gene_id, pathways, networks, elements):
    info = get_kegg_info(gene_id)
    if info is None:
        return
    pattern = r'PATHWAY|NETWORK|ELEMENT|DISEASE|DRUG_TARGET|BRITE'
    try:
        split_info = re.split(pattern, info)[1:4]
        for single_part in split_info:
            id_defs = single_part.split('\n')
            ele_pattern = r'^(N\d{5})$'
            net_pattern = r'^(nt\d{5})$'
            for id_def in id_defs:
                id_def = id_def.strip().split(' ', 1)[0].strip()
                if 'hsa' in id_def:
                    pathways.add(id_def)
                elif re.match(net_pattern, id_def):
                    networks.add(id_def)
                elif re.match(ele_pattern, id_def):
                    elements.add(id_def)
    except Exception:
        print('Error processing', gene_id)

# Initialize sets
pathways = set()
networks = set()
elements = set()

count = 0
total = len(full_dict)
for key, value in full_dict.items():
    count += 1
    get_pathways_networks_elements(value, pathways, networks, elements)
    if count % 10 == 0:
        print(f'{count} of {total} genes done, pathways: {len(pathways)}, networks: {len(networks)}, elements: {len(elements)}')

# ----------------------------------------------------------------------
# 4. For each pathway, extract its elements
# ----------------------------------------------------------------------
def pathway_to_network(pathway_id, elements):
    pathway_info = get_kegg_info(pathway_id)
    if pathway_info is None:
        return
    pattern = r'DRUG|DBLINKS|ORGANISM|GENE|COMPOUND|REFERENCE'
    if len(pathway_info.split('ELEMENT')) > 1:
        curr_elements = pathway_info.split('ELEMENT')[1]
        curr_elements = re.split(pattern, curr_elements)[0]
        curr_elements = curr_elements.split('\n')
        for element in curr_elements:
            element = element.strip(' ').split(' ', 1)[0].strip()
            ele_pattern = r'^(N\d{5})$'
            if re.match(ele_pattern, element):
                elements.add(element)

count = 0
total = len(pathways)
for pathway in list(pathways):
    pathway_to_network(pathway, elements)
    count += 1
    if count % 10 == 0:
        print(f'{count} of {total} pathways done, elements count: {len(elements)}')

# ----------------------------------------------------------------------
# 5. Get detailed data for each element
# ----------------------------------------------------------------------
def get_element_data(element_id, ppi_dict):
    ele_info = get_kegg_info(element_id)
    if ele_info is None:
        return
    small_dict = {}
    # Name
    try:
        name_part = ele_info.split('NAME')[1].split('DEFINITION')[0].strip().strip('\n')
    except IndexError:
        name_part = ""
    small_dict['name'] = name_part
    # Definition
    try:
        def_part = ele_info.split('DEFINITION')[1].split('EXPANDED')[0].strip().strip('\n')
    except IndexError:
        def_part = ""
    small_dict['definition'] = def_part
    # ID
    try:
        ele_id = ele_info.split('ENTRY')[1].split('Network')[0].strip().strip('\n')
    except IndexError:
        ele_id = element_id
    ppi_dict[ele_id] = small_dict

ppi_dict = {}
count = 0
total = len(elements)
for ele_id in list(elements):
    get_element_data(ele_id, ppi_dict)
    count += 1
    time.sleep(1)  # polite rate limiting
    if count % 50 == 0:
        print(f'{count} of {total} elements done')

# ----------------------------------------------------------------------
# 6. Save results to JSON file (relative path, no user info)
# ----------------------------------------------------------------------
output_path = "./oncogene_ppi.json"   # <-- Changed from absolute user path
with open(output_path, 'w', encoding='utf-8') as file:
    json.dump(ppi_dict, file, indent=4)

print(f"Data saved to {output_path}")